## **NoRegretsRVC**
Custom RVC voice model training environment built on Applio + Streamlit (32000 · HiFi-GAN · RMVPE).

- **Dataset Path**: `/kaggle/input/datasets/your-username/dataset`
- **Included Tabs**: samples, train, voice_blender, tts, models, settings
- **Tunnel & Auth**: Gradio + Cloudflare with auto-generated 24-byte WebUI password


## 1. Install Dependencies & Apply NoRegretsRVC Streamlit WebUI


In [ ]:
import base64
import ctypes.util
import hashlib
import os
import shutil
import subprocess
import sys
import urllib.request
from pathlib import Path
from IPython.display import clear_output

repo_dir = Path("/kaggle/working/program_ml")
repo_url = "https://github.com/IAHispano/Applio.git"

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
if repo_dir.exists() and (repo_dir / ".git").is_dir():
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1", repo_url, str(repo_dir)], check=True)

if shutil.which("fuser") is None or ctypes.util.find_library("portaudio") is None:
    subprocess.run(["apt", "update", "-y"], check=True)
    subprocess.run(["apt", "install", "-y", "portaudio19-dev", "psmisc"], check=True)

uv_bin = shutil.which("uv") or str(Path(sys.executable).with_name("uv"))
venv_python = Path("/kaggle/.venv/bin/python")
if not venv_python.exists():
    subprocess.run([uv_bin, "venv", "/kaggle/.venv", "--python", "3.12", "--seed"], check=True)
py_exec = str(venv_python)
subprocess.run([uv_bin, "pip", "install", "-q", "-r", str(repo_dir / "requirements.txt"), "--extra-index-url", "https://download.pytorch.org/whl/cu128", "--index-strategy", "unsafe-best-match", "--python", py_exec], check=True)
subprocess.run([uv_bin, "pip", "install", "-q", "streamlit>=1.40.0", "matplotlib-inline", "websockets", "--python", py_exec], check=True)
subprocess.run([uv_bin, "pip", "install", "-q", "onnxruntime-gpu", "--python", py_exec] if True else [uv_bin, "pip", "install", "-q", "onnxruntime-gpu", "--system"], check=False)
subprocess.run([py_exec, "-c", "import onnxruntime as ort; print('ONNX Providers:', ort.get_available_providers())"], check=False)

# Reset previously patched config so re-runs stay clean
if (repo_dir / ".git").is_dir():
    subprocess.run(["git", "-C", str(repo_dir), "checkout", "--", "assets/config_template.json"], check=False)

# Enforce NoRegretsRVC engine defaults (fp32 precision, Discord presence off)
import json as _cfg_json
for _cfg_rel in ("assets/config_template.json", "assets/config.json"):
    _cfg_p = repo_dir / _cfg_rel
    if _cfg_p.is_file():
        try:
            _cfg_d = _cfg_json.loads(_cfg_p.read_text(encoding="utf-8"))
            _cfg_d["precision"] = "fp32"
            _cfg_d["discord_presence"] = False
            _cfg_p.write_text(_cfg_json.dumps(_cfg_d, indent=4, ensure_ascii=False), encoding="utf-8")
        except Exception:
            pass

# Write NoRegretsRVC Streamlit WebUI and locked crimson theme
(repo_dir / ".streamlit").mkdir(parents=True, exist_ok=True)
(repo_dir / ".streamlit" / "config.toml").write_bytes(base64.b64decode("W3RoZW1lXQpiYXNlID0gImRhcmsiCnByaW1hcnlDb2xvciA9ICIjRTUxOTIzIgpiYWNrZ3JvdW5kQ29sb3IgPSAiIzBEMDQwNSIKc2Vjb25kYXJ5QmFja2dyb3VuZENvbG9yID0gIiMxNjA3MDkiCnRleHRDb2xvciA9ICIjRkRGN0Y3Igpmb250ID0gInNhbnMgc2VyaWYiCgpbc2VydmVyXQpoZWFkbGVzcyA9IHRydWUKZW5hYmxlQ09SUyA9IGZhbHNlCmVuYWJsZVhzcmZQcm90ZWN0aW9uID0gZmFsc2UKbWF4VXBsb2FkU2l6ZSA9IDIwNDgKClticm93c2VyXQpnYXRoZXJVc2FnZVN0YXRzID0gZmFsc2UK"))
(repo_dir / "streamlit_app.py").write_bytes(base64.b64decode("..."))
(repo_dir / "noregrets_turbo.py").write_bytes(base64.b64decode("ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGF0ZXhpdApmcm9tIGNvbmN1cnJlbnQuZnV0dXJlcyBpbXBvcnQgRnV0dXJlLCBUaHJlYWRQb29sRXhlY3V0b3IKaW1wb3J0IGluc3BlY3QKaW1wb3J0IGlvCmltcG9ydCBvcwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IHRocmVhZGluZwpmcm9tIHR5cGluZyBpbXBvcnQgQW55CgpfQVBQTElFRCA9IEZhbHNlCl9DQUNIRV9MT0NLID0gdGhyZWFkaW5nLkxvY2soKQpfQVJSQVlfQ0FDSEU6IGRpY3Rbc3RyLCBBbnldID0ge30KX0NBQ0hFX0JZVEVTID0gMApfQ0FDSEVfQ0hFQ0tTID0gMApfUEVORElOR19TQVZFUzogZGljdFtzdHIsIEZ1dHVyZVtOb25lXV0gPSB7fQpfU0FWRV9MT0NLID0gdGhyZWFkaW5nLkxvY2soKQpfU0FWRV9QT09MOiBUaHJlYWRQb29sRXhlY3V0b3IgfCBOb25lID0gTm9uZQoKX0RBVEFTRVRfU1VCRElSUyA9ICgic2xpY2VkX2F1ZGlvcyIsICJleHRyYWN0ZWQiLCAiZjAiLCAiZjBfdm9pY2VkIikKX01BWF9TSU5HTEVfRklMRV9CWVRFUyA9IDE2ICogMTAyNCAqIDEwMjQKX1BFUl9QUk9DRVNTX1JBTV9DQVBfQllURVMgPSAyICogMTAyNCAqIDEwMjQgKiAxMDI0Cl9NSU5fU1lTVEVNX0ZSRUVfUkFNX0JZVEVTID0gOCAqIDEwMjQgKiAxMDI0ICogMTAyNAoKCmRlZiBfcmVhZF9tZW1fYXZhaWxhYmxlX2J5dGVzKCkgLT4gaW50OgogICAgdHJ5OgogICAgICAgIHdpdGggb3BlbigiL3Byb2MvbWVtaW5mbyIsICJyIiwgZW5jb2Rpbmc9InV0Zi04IikgYXMgZmg6CiAgICAgICAgICAgIGZvciBsaW5lIGluIGZoOgogICAgICAgICAgICAgICAgaWYgbGluZS5zdGFydHN3aXRoKCJNZW1BdmFpbGFibGU6Iik6CiAgICAgICAgICAgICAgICAgICAga2IgPSBpbnQobGluZS5zcGxpdCgpWzFdKQogICAgICAgICAgICAgICAgICAgIHJldHVybiBrYiAqIDEwMjQKICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgcGFzcwogICAgcmV0dXJuIDAKCgpkZWYgX2hhc19zYWZlX3JhbShpbmNvbWluZ19ieXRlczogaW50KSAtPiBib29sOgogICAgZ2xvYmFsIF9DQUNIRV9DSEVDS1MKICAgIGlmIF9DQUNIRV9CWVRFUyArIGluY29taW5nX2J5dGVzID4gX1BFUl9QUk9DRVNTX1JBTV9DQVBfQllURVM6CiAgICAgICAgcmV0dXJuIEZhbHNlCiAgICBfQ0FDSEVfQ0hFQ0tTICs9IDEKICAgIGlmIF9DQUNIRV9DSEVDS1MgPT0gMSBvciBfQ0FDSEVfQ0hFQ0tTICUgMjQgPT0gMDoKICAgICAgICBhdmFpbCA9IF9yZWFkX21lbV9hdmFpbGFibGVfYnl0ZXMoKQogICAgICAgIGlmIGF2YWlsID4gMCBhbmQgYXZhaWwgLSBpbmNvbWluZ19ieXRlcyA8IF9NSU5fU1lTVEVNX0ZSRUVfUkFNX0JZVEVTOgogICAgICAgICAgICByZXR1cm4gRmFsc2UKICAgIHJldHVybiBUcnVlCgoKZGVmIF9pc19jYWNoZWFibGVfZGF0YXNldF9wYXRoKHBhdGg6IEFueSkgLT4gc3RyIHwgTm9uZToKICAgIGlmIG5vdCBpc2luc3RhbmNlKHBhdGgsIChzdHIsIG9zLlBhdGhMaWtlKSk6CiAgICAgICAgcmV0dXJuIE5vbmUKICAgIHJhdyA9IG9zLmZzcGF0aChwYXRoKQogICAgbm9ybSA9IHJhdy5yZXBsYWNlKCJcXCIsICIvIikKICAgIGlmICIvbG9ncy8iIG5vdCBpbiBub3JtIGFuZCBub3Qgbm9ybS5zdGFydHN3aXRoKCJsb2dzLyIpOgogICAgICAgIHJldHVybiBOb25lCiAgICBpZiBub3QgYW55KGYiL3twYXJ0fS8iIGluIG5vcm0gb3Igbm9ybS5zdGFydHN3aXRoKGYie3BhcnR9LyIpIGZvciBwYXJ0IGluIF9EQVRBU0VUX1NVQkRJUlMpOgogICAgICAgIHJldHVybiBOb25lCiAgICB0cnk6CiAgICAgICAgaWYgb3MucGF0aC5nZXRzaXplKHJhdykgPiBfTUFYX1NJTkdMRV9GSUxFX0JZVEVTOgogICAgICAgICAgICByZXR1cm4gTm9uZQogICAgZXhjZXB0IE9TRXJyb3I6CiAgICAgICAgcmV0dXJuIE5vbmUKICAgIHJldHVybiByYXcKCgpkZWYgX2FwcGx5X2N1ZGFfdHVuaW5nKCkgLT4gTm9uZToKICAgIGltcG9ydCB0b3JjaAoKICAgIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCk6CiAgICAgICAgdG9yY2guYmFja2VuZHMuY3Vkbm4uZW5hYmxlZCA9IFRydWUKICAgICAgICB0b3JjaC5iYWNrZW5kcy5jdWRubi5iZW5jaG1hcmsgPSBUcnVlCgoKZGVmIF9wYXRjaF9kYXRhbG9hZGVyKCkgLT4gTm9uZToKICAgIGltcG9ydCB0b3JjaC51dGlscy5kYXRhCgogICAgb3JpZ19pbml0ID0gdG9yY2gudXRpbHMuZGF0YS5EYXRhTG9hZGVyLl9faW5pdF9fCiAgICBpZiBnZXRhdHRyKG9yaWdfaW5pdCwgIl9ub3JlZ3JldHNfcGF0Y2hlZCIsIEZhbHNlKToKICAgICAgICByZXR1cm4KCiAgICBjcHVfY2FwID0gbWF4KDIsIG1pbig0LCBvcy5jcHVfY291bnQoKSBvciAyKSkKCiAgICBkZWYgZmFzdF9pbml0KHNlbGY6IEFueSwgKmFyZ3M6IEFueSwgKiprd2FyZ3M6IEFueSkgLT4gQW55OgogICAgICAgIG51bV93b3JrZXJzID0ga3dhcmdzLmdldCgibnVtX3dvcmtlcnMiLCAwKQogICAgICAgIGlmICJudW1fd29ya2VycyIgbm90IGluIGt3YXJncyBhbmQgbGVuKGFyZ3MpID49IDY6CiAgICAgICAgICAgIG51bV93b3JrZXJzID0gYXJnc1s1XQogICAgICAgIGlmIGlzaW5zdGFuY2UobnVtX3dvcmtlcnMsIGludCkgYW5kIG51bV93b3JrZXJzID4gMDoKICAgICAgICAgICAgaWYgIm51bV93b3JrZXJzIiBpbiBrd2FyZ3MgYW5kIG51bV93b3JrZXJzID4gY3B1X2NhcDoKICAgICAgICAgICAgICAgIGt3YXJnc1sibnVtX3dvcmtlcnMiXSA9IGNwdV9jYXAKICAgICAgICAgICAga3dhcmdzLnNldGRlZmF1bHQoInBpbl9tZW1vcnkiLCBUcnVlKQogICAgICAgICAgICBrd2FyZ3Muc2V0ZGVmYXVsdCgicGVyc2lzdGVudF93b3JrZXJzIiwgVHJ1ZSkKICAgICAgICAgICAga3dhcmdzLnNldGRlZmF1bHQoInByZWZldGNoX2ZhY3RvciIsIDQpCiAgICAgICAgdHJ5OgogICAgICAgICAgICByZXR1cm4gb3JpZ19pbml0KHNlbGYsICphcmdzLCAqKmt3YXJncykKICAgICAgICBleGNlcHQgKFR5cGVFcnJvciwgVmFsdWVFcnJvcik6CiAgICAgICAgICAgIGt3YXJncy5wb3AoInBlcnNpc3RlbnRfd29ya2VycyIsIE5vbmUpCiAgICAgICAgICAgIGt3YXJncy5wb3AoInByZWZldGNoX2ZhY3RvciIsIE5vbmUpCiAgICAgICAgICAgIGt3YXJncy5wb3AoInBpbl9tZW1vcnkiLCBOb25lKQogICAgICAgICAgICByZXR1cm4gb3JpZ19pbml0KHNlbGYsICphcmdzLCAqKmt3YXJncykKCiAgICBmYXN0X2luaXQuX25vcmVncmV0c19wYXRjaGVkID0gVHJ1ZSAgIyB0eXBlOiBpZ25vcmVbYXR0ci1kZWZpbmVkXQogICAgdG9yY2gudXRpbHMuZGF0YS5EYXRhTG9hZGVyLl9faW5pdF9fID0gZmFzdF9pbml0ICAjIHR5cGU6IGlnbm9yZVttZXRob2QtYXNzaWduXQoKCmRlZiBfcGF0Y2hfb3B0aW1pemVyKCkgLT4gTm9uZToKICAgIGltcG9ydCB0b3JjaAoKICAgIG9yaWdfYWRhbXdfaW5pdCA9IHRvcmNoLm9wdGltLkFkYW1XLl9faW5pdF9fCiAgICBpZiBub3QgZ2V0YXR0cihvcmlnX2FkYW13X2luaXQsICJfbm9yZWdyZXRzX3BhdGNoZWQiLCBGYWxzZSk6CiAgICAgICAgc3VwcG9ydHNfZnVzZWQgPSAiZnVzZWQiIGluIGluc3BlY3Quc2lnbmF0dXJlKG9yaWdfYWRhbXdfaW5pdCkucGFyYW1ldGVycwoKICAgICAgICBkZWYgZmFzdF9hZGFtd19pbml0KHNlbGY6IEFueSwgcGFyYW1zOiBBbnksICphcmdzOiBBbnksICoqa3dhcmdzOiBBbnkpIC0+IEFueToKICAgICAgICAgICAgX2FwcGx5X2N1ZGFfdHVuaW5nKCkKICAgICAgICAgICAgaWYgc3VwcG9ydHNfZnVzZWQgYW5kIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgYW5kICJmdXNlZCIgbm90IGluIGt3YXJnczoKICAgICAgICAgICAgICAgIGlmIG5vdCBpc2luc3RhbmNlKHBhcmFtcywgKGxpc3QsIHR1cGxlKSk6CiAgICAgICAgICAgICAgICAgICAgcGFyYW1zID0gbGlzdChwYXJhbXMpCiAgICAgICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICAgICAgcmV0dXJuIG9yaWdfYWRhbXdfaW5pdChzZWxmLCBwYXJhbXMsICphcmdzLCBmdXNlZD1UcnVlLCAqKmt3YXJncykKICAgICAgICAgICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgICAgICAgICAgcGFzcwogICAgICAgICAgICByZXR1cm4gb3JpZ19hZGFtd19pbml0KHNlbGYsIHBhcmFtcywgKmFyZ3MsICoqa3dhcmdzKQoKICAgICAgICBmYXN0X2FkYW13X2luaXQuX25vcmVncmV0c19wYXRjaGVkID0gVHJ1ZSAgIyB0eXBlOiBpZ25vcmVbYXR0ci1kZWZpbmVkXQogICAgICAgIHRvcmNoLm9wdGltLkFkYW1XLl9faW5pdF9fID0gZmFzdF9hZGFtd19pbml0ICAjIHR5cGU6IGlnbm9yZVttZXRob2QtYXNzaWduXQoKICAgIG9yaWdfemVyb19ncmFkID0gdG9yY2gub3B0aW0uT3B0aW1pemVyLnplcm9fZ3JhZAogICAgaWYgbm90IGdldGF0dHIob3JpZ196ZXJvX2dyYWQsICJfbm9yZWdyZXRzX3BhdGNoZWQiLCBGYWxzZSk6CiAgICAgICAgZGVmIGZhc3RfemVyb19ncmFkKHNlbGY6IEFueSwgc2V0X3RvX25vbmU6IGJvb2wgPSBUcnVlKSAtPiBBbnk6CiAgICAgICAgICAgIHJldHVybiBvcmlnX3plcm9fZ3JhZChzZWxmLCBzZXRfdG9fbm9uZT1zZXRfdG9fbm9uZSkKCiAgICAgICAgZmFzdF96ZXJvX2dyYWQuX25vcmVncmV0c19wYXRjaGVkID0gVHJ1ZSAgIyB0eXBlOiBpZ25vcmVbYXR0ci1kZWZpbmVkXQogICAgICAgIHRvcmNoLm9wdGltLk9wdGltaXplci56ZXJvX2dyYWQgPSBmYXN0X3plcm9fZ3JhZCAgIyB0eXBlOiBpZ25vcmVbbWV0aG9kLWFzc2lnbl0KCgpkZWYgX3BhdGNoX2RkcCgpIC0+IE5vbmU6CiAgICBpbXBvcnQgdG9yY2gubm4ucGFyYWxsZWwKCiAgICBkZHBfY2xzID0gdG9yY2gubm4ucGFyYWxsZWwuRGlzdHJpYnV0ZWREYXRhUGFyYWxsZWwKICAgIG9yaWdfZGRwX2luaXQgPSBkZHBfY2xzLl9faW5pdF9fCiAgICBpZiBnZXRhdHRyKG9yaWdfZGRwX2luaXQsICJfbm9yZWdyZXRzX3BhdGNoZWQiLCBGYWxzZSk6CiAgICAgICAgcmV0dXJuCgogICAgZGVmIGZhc3RfZGRwX2luaXQoc2VsZjogQW55LCAqYXJnczogQW55LCAqKmt3YXJnczogQW55KSAtPiBBbnk6CiAgICAgICAgX2FwcGx5X2N1ZGFfdHVuaW5nKCkKICAgICAgICBrd2FyZ3Muc2V0ZGVmYXVsdCgiZ3JhZGllbnRfYXNfYnVja2V0X3ZpZXciLCBUcnVlKQogICAgICAgIHRyeToKICAgICAgICAgICAgcmV0dXJuIG9yaWdfZGRwX2luaXQoc2VsZiwgKmFyZ3MsICoqa3dhcmdzKQogICAgICAgIGV4Y2VwdCBUeXBlRXJyb3I6CiAgICAgICAgICAgIGt3YXJncy5wb3AoImdyYWRpZW50X2FzX2J1Y2tldF92aWV3IiwgTm9uZSkKICAgICAgICAgICAgcmV0dXJuIG9yaWdfZGRwX2luaXQoc2VsZiwgKmFyZ3MsICoqa3dhcmdzKQoKICAgIGZhc3RfZGRwX2luaXQuX25vcmVncmV0c19wYXRjaGVkID0gVHJ1ZSAgIyB0eXBlOiBpZ25vcmVbYXR0ci1kZWZpbmVkXQogICAgZGRwX2Nscy5fX2luaXRfXyA9IGZhc3RfZGRwX2luaXQgICMgdHlwZTogaWdub3JlW21ldGhvZC1hc3NpZ25dCgoKZGVmIF9wYXRjaF90ZW5zb3Jib2FyZF9pbWFnZXMoKSAtPiBOb25lOgogICAgdHJ5OgogICAgICAgIGZyb20gdG9yY2gudXRpbHMudGVuc29yYm9hcmQgaW1wb3J0IFN1bW1hcnlXcml0ZXIKICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgcmV0dXJuCgogICAgZGVmIF9za2lwX2ltYWdlKCphcmdzOiBBbnksICoqa3dhcmdzOiBBbnkpIC0+IE5vbmU6CiAgICAgICAgcmV0dXJuIE5vbmUKCiAgICBmb3IgbWV0aG9kIGluICgiYWRkX2ltYWdlIiwgImFkZF9pbWFnZXMiLCAiYWRkX2ZpZ3VyZSIpOgogICAgICAgIGlmIGhhc2F0dHIoU3VtbWFyeVdyaXRlciwgbWV0aG9kKToKICAgICAgICAgICAgc2V0YXR0cihTdW1tYXJ5V3JpdGVyLCBtZXRob2QsIF9za2lwX2ltYWdlKQoKCmRlZiBfcGF0Y2hfcmFtX2NhY2hlKCkgLT4gTm9uZToKICAgIGltcG9ydCBudW1weSBhcyBucAoKICAgIG9yaWdfbnBfbG9hZCA9IG5wLmxvYWQKICAgIGlmIG5vdCBnZXRhdHRyKG9yaWdfbnBfbG9hZCwgIl9ub3JlZ3JldHNfcGF0Y2hlZCIsIEZhbHNlKToKICAgICAgICBkZWYgZmFzdF9ucF9sb2FkKGZpbGU6IEFueSwgKmFyZ3M6IEFueSwgKiprd2FyZ3M6IEFueSkgLT4gQW55OgogICAgICAgICAgICBnbG9iYWwgX0NBQ0hFX0JZVEVTCiAgICAgICAgICAgIGlmIGFyZ3Mgb3Iga3dhcmdzOgogICAgICAgICAgICAgICAgcmV0dXJuIG9yaWdfbnBfbG9hZChmaWxlLCAqYXJncywgKiprd2FyZ3MpCiAgICAgICAgICAgIGtleSA9IF9pc19jYWNoZWFibGVfZGF0YXNldF9wYXRoKGZpbGUpCiAgICAgICAgICAgIGlmIGtleSBpcyBOb25lOgogICAgICAgICAgICAgICAgcmV0dXJuIG9yaWdfbnBfbG9hZChmaWxlKQogICAgICAgICAgICBjYWNoZV9rZXkgPSBmIm5wOjp7a2V5fSIKICAgICAgICAgICAgd2l0aCBfQ0FDSEVfTE9DSzoKICAgICAgICAgICAgICAgIGNhY2hlZCA9IF9BUlJBWV9DQUNIRS5nZXQoY2FjaGVfa2V5KQogICAgICAgICAgICBpZiBjYWNoZWQgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICByZXR1cm4gY2FjaGVkLmNvcHkoKQogICAgICAgICAgICBhcnIgPSBvcmlnX25wX2xvYWQoZmlsZSkKICAgICAgICAgICAgbmJ5dGVzID0gaW50KGdldGF0dHIoYXJyLCAibmJ5dGVzIiwgMCkpCiAgICAgICAgICAgIGlmIG5ieXRlcyA+IDAgYW5kIGlzaW5zdGFuY2UoYXJyLCBucC5uZGFycmF5KToKICAgICAgICAgICAgICAgIHdpdGggX0NBQ0hFX0xPQ0s6CiAgICAgICAgICAgICAgICAgICAgaWYgY2FjaGVfa2V5IG5vdCBpbiBfQVJSQVlfQ0FDSEUgYW5kIF9oYXNfc2FmZV9yYW0obmJ5dGVzKToKICAgICAgICAgICAgICAgICAgICAgICAgX0FSUkFZX0NBQ0hFW2NhY2hlX2tleV0gPSBhcnIuY29weSgpCiAgICAgICAgICAgICAgICAgICAgICAgIF9DQUNIRV9CWVRFUyArPSBuYnl0ZXMKICAgICAgICAgICAgcmV0dXJuIGFycgoKICAgICAgICBmYXN0X25wX2xvYWQuX25vcmVncmV0c19wYXRjaGVkID0gVHJ1ZSAgIyB0eXBlOiBpZ25vcmVbYXR0ci1kZWZpbmVkXQogICAgICAgIG5wLmxvYWQgPSBmYXN0X25wX2xvYWQgICMgdHlwZTogaWdub3JlW2Fzc2lnbm1lbnRdCgogICAgdHJ5OgogICAgICAgIGltcG9ydCBzY2lweS5pby53YXZmaWxlIGFzIHdhdmZpbGUKCiAgICAgICAgb3JpZ193YXZfcmVhZCA9IHdhdmZpbGUucmVhZAogICAgICAgIGlmIG5vdCBnZXRhdHRyKG9yaWdfd2F2X3JlYWQsICJfbm9yZWdyZXRzX3BhdGNoZWQiLCBGYWxzZSk6CiAgICAgICAgICAgIGRlZiBmYXN0X3dhdl9yZWFkKGZpbGVuYW1lOiBBbnksICphcmdzOiBBbnksICoqa3dhcmdzOiBBbnkpIC0+IEFueToKICAgICAgICAgICAgICAgIGdsb2JhbCBfQ0FDSEVfQllURVMKICAgICAgICAgICAgICAgIGlmIGFyZ3Mgb3Iga3dhcmdzOgogICAgICAgICAgICAgICAgICAgIHJldHVybiBvcmlnX3dhdl9yZWFkKGZpbGVuYW1lLCAqYXJncywgKiprd2FyZ3MpCiAgICAgICAgICAgICAgICBrZXkgPSBfaXNfY2FjaGVhYmxlX2RhdGFzZXRfcGF0aChmaWxlbmFtZSkKICAgICAgICAgICAgICAgIGlmIGtleSBpcyBOb25lOgogICAgICAgICAgICAgICAgICAgIHJldHVybiBvcmlnX3dhdl9yZWFkKGZpbGVuYW1lKQogICAgICAgICAgICAgICAgY2FjaGVfa2V5ID0gZiJ3YXY6OntrZXl9IgogICAgICAgICAgICAgICAgd2l0aCBfQ0FDSEVfTE9DSzoKICAgICAgICAgICAgICAgICAgICBjYWNoZWQgPSBfQVJSQVlfQ0FDSEUuZ2V0KGNhY2hlX2tleSkKICAgICAgICAgICAgICAgIGlmIGNhY2hlZCBpcyBub3QgTm9uZToKICAgICAgICAgICAgICAgICAgICBzciwgZGF0YSA9IGNhY2hlZAogICAgICAgICAgICAgICAgICAgIHJldHVybiBzciwgZGF0YS5jb3B5KCkKICAgICAgICAgICAgICAgIHNyLCBkYXRhID0gb3JpZ193YXZfcmVhZChmaWxlbmFtZSkKICAgICAgICAgICAgICAgIG5ieXRlcyA9IGludChnZXRhdHRyKGRhdGEsICJuYnl0ZXMiLCAwKSkKICAgICAgICAgICAgICAgIGlmIG5ieXRlcyA+IDA6CiAgICAgICAgICAgICAgICAgICAgd2l0aCBfQ0FDSEVfTE9DSzoKICAgICAgICAgICAgICAgICAgICAgICAgaWYgY2FjaGVfa2V5IG5vdCBpbiBfQVJSQVlfQ0FDSEUgYW5kIF9oYXNfc2FmZV9yYW0obmJ5dGVzKToKICAgICAgICAgICAgICAgICAgICAgICAgICAgIF9BUlJBWV9DQUNIRVtjYWNoZV9rZXldID0gKHNyLCBkYXRhLmNvcHkoKSkKICAgICAgICAgICAgICAgICAgICAgICAgICAgIF9DQUNIRV9CWVRFUyArPSBuYnl0ZXMKICAgICAgICAgICAgICAgIHJldHVybiBzciwgZGF0YQoKICAgICAgICAgICAgZmFzdF93YXZfcmVhZC5fbm9yZWdyZXRzX3BhdGNoZWQgPSBUcnVlICAjIHR5cGU6IGlnbm9yZVthdHRyLWRlZmluZWRdCiAgICAgICAgICAgIHdhdmZpbGUucmVhZCA9IGZhc3Rfd2F2X3JlYWQgICMgdHlwZTogaWdub3JlW2Fzc2lnbm1lbnRdCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIHBhc3MKCiAgICB0cnk6CiAgICAgICAgaW1wb3J0IHNvdW5kZmlsZSBhcyBzZgoKICAgICAgICBvcmlnX3NmX3JlYWQgPSBzZi5yZWFkCiAgICAgICAgaWYgbm90IGdldGF0dHIob3JpZ19zZl9yZWFkLCAiX25vcmVncmV0c19wYXRjaGVkIiwgRmFsc2UpOgogICAgICAgICAgICBkZWYgZmFzdF9zZl9yZWFkKGZpbGU6IEFueSwgKmFyZ3M6IEFueSwgKiprd2FyZ3M6IEFueSkgLT4gQW55OgogICAgICAgICAgICAgICAgZ2xvYmFsIF9DQUNIRV9CWVRFUwogICAgICAgICAgICAgICAgaWYgYXJncyBvciBrd2FyZ3M6CiAgICAgICAgICAgICAgICAgICAgcmV0dXJuIG9yaWdfc2ZfcmVhZChmaWxlLCAqYXJncywgKiprd2FyZ3MpCiAgICAgICAgICAgICAgICBrZXkgPSBfaXNfY2FjaGVhYmxlX2RhdGFzZXRfcGF0aChmaWxlKQogICAgICAgICAgICAgICAgaWYga2V5IGlzIE5vbmU6CiAgICAgICAgICAgICAgICAgICAgcmV0dXJuIG9yaWdfc2ZfcmVhZChmaWxlKQogICAgICAgICAgICAgICAgY2FjaGVfa2V5ID0gZiJzZjo6e2tleX0iCiAgICAgICAgICAgICAgICB3aXRoIF9DQUNIRV9MT0NLOgogICAgICAgICAgICAgICAgICAgIGNhY2hlZCA9IF9BUlJBWV9DQUNIRS5nZXQoY2FjaGVfa2V5KQogICAgICAgICAgICAgICAgaWYgY2FjaGVkIGlzIG5vdCBOb25lOgogICAgICAgICAgICAgICAgICAgIGRhdGEsIHNyID0gY2FjaGVkCiAgICAgICAgICAgICAgICAgICAgcmV0dXJuIGRhdGEuY29weSgpLCBzcgogICAgICAgICAgICAgICAgZGF0YSwgc3IgPSBvcmlnX3NmX3JlYWQoZmlsZSkKICAgICAgICAgICAgICAgIG5ieXRlcyA9IGludChnZXRhdHRyKGRhdGEsICJuYnl0ZXMiLCAwKSkKICAgICAgICAgICAgICAgIGlmIG5ieXRlcyA+IDA6CiAgICAgICAgICAgICAgICAgICAgd2l0aCBfQ0FDSEVfTE9DSzoKICAgICAgICAgICAgICAgICAgICAgICAgaWYgY2FjaGVfa2V5IG5vdCBpbiBfQVJSQVlfQ0FDSEUgYW5kIF9oYXNfc2FmZV9yYW0obmJ5dGVzKToKICAgICAgICAgICAgICAgICAgICAgICAgICAgIF9BUlJBWV9DQUNIRVtjYWNoZV9rZXldID0gKGRhdGEuY29weSgpLCBzcikKICAgICAgICAgICAgICAgICAgICAgICAgICAgIF9DQUNIRV9CWVRFUyArPSBuYnl0ZXMKICAgICAgICAgICAgICAgIHJldHVybiBkYXRhLCBzcgoKICAgICAgICAgICAgZmFzdF9zZl9yZWFkLl9ub3JlZ3JldHNfcGF0Y2hlZCA9IFRydWUgICMgdHlwZTogaWdub3JlW2F0dHItZGVmaW5lZF0KICAgICAgICAgICAgc2YucmVhZCA9IGZhc3Rfc2ZfcmVhZCAgIyB0eXBlOiBpZ25vcmVbYXNzaWdubWVudF0KICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgcGFzcwoKCmRlZiBfd3JpdGVfYXRvbWljKHRhcmdldDogUGF0aCwgcGF5bG9hZDogYnl0ZXMpIC0+IE5vbmU6CiAgICB0YXJnZXQucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHRtcCA9IHRhcmdldC53aXRoX25hbWUoZiIue3RhcmdldC5uYW1lfS50bXAiKQogICAgdHJ5OgogICAgICAgIHdpdGggb3Blbih0bXAsICJ3YiIpIGFzIGZoOgogICAgICAgICAgICBmaC53cml0ZShwYXlsb2FkKQogICAgICAgIG9zLnJlcGxhY2UodG1wLCB0YXJnZXQpCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIHRyeToKICAgICAgICAgICAgaWYgdG1wLmV4aXN0cygpOgogICAgICAgICAgICAgICAgdG1wLnVubGluaygpCiAgICAgICAgZXhjZXB0IE9TRXJyb3I6CiAgICAgICAgICAgIHBhc3MKICAgICAgICByYWlzZQoKCmRlZiBfcGF0Y2hfYXN5bmNfc2F2ZSgpIC0+IE5vbmU6CiAgICBnbG9iYWwgX1NBVkVfUE9PTAogICAgaW1wb3J0IHRvcmNoCgogICAgb3JpZ19zYXZlID0gdG9yY2guc2F2ZQogICAgb3JpZ19sb2FkID0gdG9yY2gubG9hZAogICAgaWYgZ2V0YXR0cihvcmlnX3NhdmUsICJfbm9yZWdyZXRzX3BhdGNoZWQiLCBGYWxzZSk6CiAgICAgICAgcmV0dXJuCgogICAgX1NBVkVfUE9PTCA9IFRocmVhZFBvb2xFeGVjdXRvcihtYXhfd29ya2Vycz0xLCB0aHJlYWRfbmFtZV9wcmVmaXg9Im5yLWNrcHQiKQogICAgYXRleGl0LnJlZ2lzdGVyKGxhbWJkYTogX1NBVkVfUE9PTC5zaHV0ZG93bih3YWl0PVRydWUpIGlmIF9TQVZFX1BPT0wgZWxzZSBOb25lKQoKICAgIGRlZiBmYXN0X3NhdmUob2JqOiBBbnksIGY6IEFueSwgKmFyZ3M6IEFueSwgKiprd2FyZ3M6IEFueSkgLT4gQW55OgogICAgICAgIGlmICgKICAgICAgICAgICAgX1NBVkVfUE9PTCBpcyBub3QgTm9uZQogICAgICAgICAgICBhbmQgaXNpbnN0YW5jZShmLCAoc3RyLCBvcy5QYXRoTGlrZSkpCiAgICAgICAgICAgIGFuZCBub3QgYXJncwogICAgICAgICAgICBhbmQgbm90IGt3YXJncwogICAgICAgICk6CiAgICAgICAgICAgIHJhdyA9IG9zLmZzcGF0aChmKQogICAgICAgICAgICBub3JtID0gcmF3LnJlcGxhY2UoIlxcIiwgIi8iKQogICAgICAgICAgICBpZiBub3JtLmVuZHN3aXRoKCIucHRoIikgYW5kICgiL2xvZ3MvIiBpbiBub3JtIG9yIG5vcm0uc3RhcnRzd2l0aCgibG9ncy8iKSk6CiAgICAgICAgICAgICAgICBidWYgPSBpby5CeXRlc0lPKCkKICAgICAgICAgICAgICAgIG9yaWdfc2F2ZShvYmosIGJ1ZikKICAgICAgICAgICAgICAgIHRhcmdldCA9IFBhdGgocmF3KS5yZXNvbHZlKCkKICAgICAgICAgICAgICAgIGZ1dCA9IF9TQVZFX1BPT0wuc3VibWl0KF93cml0ZV9hdG9taWMsIHRhcmdldCwgYnVmLmdldHZhbHVlKCkpCiAgICAgICAgICAgICAgICB3aXRoIF9TQVZFX0xPQ0s6CiAgICAgICAgICAgICAgICAgICAgX1BFTkRJTkdfU0FWRVNbc3RyKHRhcmdldCldID0gZnV0CiAgICAgICAgICAgICAgICByZXR1cm4gTm9uZQogICAgICAgIHJldHVybiBvcmlnX3NhdmUob2JqLCBmLCAqYXJncywgKiprd2FyZ3MpCgogICAgZGVmIHNhZmVfbG9hZChmOiBBbnksICphcmdzOiBBbnksICoqa3dhcmdzOiBBbnkpIC0+IEFueToKICAgICAgICBpZiBpc2luc3RhbmNlKGYsIChzdHIsIG9zLlBhdGhMaWtlKSk6CiAgICAgICAgICAgIGtleSA9IHN0cihQYXRoKG9zLmZzcGF0aChmKSkucmVzb2x2ZSgpKQogICAgICAgICAgICB3aXRoIF9TQVZFX0xPQ0s6CiAgICAgICAgICAgICAgICBmdXQgPSBfUEVORElOR19TQVZFUy5nZXQoa2V5KQogICAgICAgICAgICBpZiBmdXQgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICBmdXQucmVzdWx0KCkKICAgICAgICByZXR1cm4gb3JpZ19sb2FkKGYsICphcmdzLCAqKmt3YXJncykKCiAgICBmYXN0X3NhdmUuX25vcmVncmV0c19wYXRjaGVkID0gVHJ1ZSAgIyB0eXBlOiBpZ25vcmVbYXR0ci1kZWZpbmVkXQogICAgdG9yY2guc2F2ZSA9IGZhc3Rfc2F2ZSAgIyB0eXBlOiBpZ25vcmVbYXNzaWdubWVudF0KICAgIHRvcmNoLmxvYWQgPSBzYWZlX2xvYWQgICMgdHlwZTogaWdub3JlW2Fzc2lnbm1lbnRdCgoKZGVmIGFwcGx5KCkgLT4gTm9uZToKICAgIGdsb2JhbCBfQVBQTElFRAogICAgaWYgX0FQUExJRUQ6CiAgICAgICAgcmV0dXJuCiAgICBfQVBQTElFRCA9IFRydWUKICAgIGZvciBzdGVwIGluICgKICAgICAgICBfYXBwbHlfY3VkYV90dW5pbmcsCiAgICAgICAgX3BhdGNoX2RhdGFsb2FkZXIsCiAgICAgICAgX3BhdGNoX29wdGltaXplciwKICAgICAgICBfcGF0Y2hfZGRwLAogICAgICAgIF9wYXRjaF90ZW5zb3Jib2FyZF9pbWFnZXMsCiAgICAgICAgX3BhdGNoX3JhbV9jYWNoZSwKICAgICAgICBfcGF0Y2hfYXN5bmNfc2F2ZSwKICAgICk6CiAgICAgICAgdHJ5OgogICAgICAgICAgICBzdGVwKCkKICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICBwYXNzCg=="))
train_py = repo_dir / "rvc" / "train" / "train.py"
if train_py.is_file():
    _src = train_py.read_text(encoding="utf-8", errors="replace")
    if "noregrets_turbo.apply()" not in _src:
        _lines = _src.splitlines(keepends=True)
        _ins = 0
        for _i, _ln in enumerate(_lines[:40]):
            if _ln.startswith("from __future__ import"):
                _ins = _i + 1
        _hook = "import os as _nr_os, sys as _nr_sys\n_nr_sys.path.insert(0, _nr_os.getcwd())\ntry:\n    import noregrets_turbo\n    noregrets_turbo.apply()\nexcept Exception:\n    pass\n"
        _lines.insert(_ins, _hook)
        train_py.write_text("".join(_lines), encoding="utf-8")

# Best-effort prefetch of the 6 Free Samples WAVs from https://github.com/HunterThatsMe/NoRegretsRVC
fs_dir = repo_dir / "assets" / "free_samples"
au_dir = repo_dir / "assets" / "audios"
fs_dir.mkdir(parents=True, exist_ok=True)
au_dir.mkdir(parents=True, exist_ok=True)
for _wav_name in (
    "MaleSpeakingTest-1_931b5ffd.wav",
    "MaleSpeakingTest-2_c2e6e7fe.wav",
    "MaleSpeakingTest-3_d78e8233.wav",
    "FemaleSpeakingTest-1_bb3d674f.wav",
    "FemaleSpeakingTest-2_d1823132.wav",
    "FemaleSpeakingTest-3_458bd4fc.wav",
):
    _dst_fs = fs_dir / _wav_name
    if not _dst_fs.is_file():
        try:
            urllib.request.urlretrieve(f"https://raw.githubusercontent.com/HunterThatsMe/NoRegretsRVC/main/public/assets/aistudio/{_wav_name}", _dst_fs)
            shutil.copy2(_dst_fs, au_dir / _wav_name)
        except Exception:
            pass

os.chdir(repo_dir)
prereq_cmd = [py_exec, "core.py", "prerequisites", "--models", "--exe", "--pretraineds-hifigan"]
subprocess.run(prereq_cmd, check=True)

# Pre-boot Model & Custom Pretrains Downloader (HF repo auto-discovery, ZIP & G/D .pth)
import json as _json
import urllib.parse
import zipfile as _zipfile
boot_downloads = [
    {
        "name": "RMVPE Pitch Model (Default)",
        "url": "https://huggingface.co/IAHispano/Applio/resolve/main/Resources/predictors/rmvpe.pt",
        "category": "pretrained"
    },
    {
        "name": "ContentVec Embedder",
        "url": "https://huggingface.co/IAHispano/Applio/resolve/main/Resources/embedders/contentvec/pytorch_model.bin",
        "category": "embedder"
    }
]
dl_dir = repo_dir / "logs" / "downloads"
custom_pretrain_dir = repo_dir / "rvc" / "models" / "pretraineds" / "custom"
dl_dir.mkdir(parents=True, exist_ok=True)
custom_pretrain_dir.mkdir(parents=True, exist_ok=True)
def _download_hf_repo_pretrains(_repo_url, _dst_dir):
    _parts = [p for p in urllib.parse.urlsplit(_repo_url).path.strip("/").split("/") if p]
    if len(_parts) < 2:
        return
    _owner_repo = f"{_parts[0]}/{_parts[1]}"
    _api = f"https://huggingface.co/api/models/{_owner_repo}/tree/main"
    with urllib.request.urlopen(_api, timeout=30) as _resp:
        _tree = _json.loads(_resp.read().decode("utf-8"))
    for _node in _tree:
        _fpath = _node.get("path", "")
        if _fpath.lower().endswith(".pth"):
            _fname = Path(_fpath).name
            _out = _dst_dir / _fname
            if not _out.is_file():
                _file_url = f"https://huggingface.co/{_owner_repo}/resolve/main/{urllib.parse.quote(_fpath)}?download=true"
                print("Downloading discovered pretrain", _fname, "->", _out)
                urllib.request.urlretrieve(_file_url, _out)
for entry in boot_downloads:
    _url = entry["url"].strip()
    _cat = entry.get("category", "voice")
    _target_dir = custom_pretrain_dir if _cat == "pretrained" else dl_dir
    _parsed_path = urllib.parse.urlsplit(_url).path
    _raw_name = urllib.parse.unquote(_parsed_path.split("/")[-1]) or entry["name"]
    try:
        if _cat == "pretrained" and "huggingface.co/" in _url.lower() and "/resolve/" not in _url and not _parsed_path.lower().endswith((".pth", ".zip")):
            _download_hf_repo_pretrains(_url, _target_dir)
        elif _cat == "pretrained" and _parsed_path.lower().endswith(".zip"):
            _zip_tmp = _target_dir / (_raw_name if _raw_name.endswith(".zip") else "custom_pretrains.zip")
            print("Downloading pretrained ZIP", _url, "->", _zip_tmp)
            urllib.request.urlretrieve(_url, _zip_tmp)
            with _zipfile.ZipFile(_zip_tmp, "r") as _zf:
                for _member in _zf.namelist():
                    if not _member.endswith("/") and Path(_member).name and _member.lower().endswith(".pth"):
                        _out_pth = _target_dir / Path(_member).name
                        _out_pth.write_bytes(_zf.read(_member))
                        print("Extracted", _out_pth.name)
            _zip_tmp.unlink(missing_ok=True)
        else:
            _dst = _target_dir / _raw_name
            if not _dst.is_file():
                print("Downloading", entry["name"], "->", _dst)
                urllib.request.urlretrieve(_url, _dst)
    except Exception as _dl_err:
        print("Skipping unreachable download", _url, ":", _dl_err)

cloudflared_path = repo_dir.parent / "cloudflared"
cloudflared_url = "https://github.com/cloudflare/cloudflared/releases/download/2026.9.0/cloudflared-linux-amd64"
cloudflared_sha256 = "53b7a7a5420d188758d24341294acb0d1bca54296548ac05e38811a694ac6134"
if not (cloudflared_path.is_file() and hashlib.sha256(cloudflared_path.read_bytes()).hexdigest() == cloudflared_sha256):
    tmp = cloudflared_path.with_suffix(".download")
    with urllib.request.urlopen(cloudflared_url, timeout=120) as resp, open(tmp, "wb") as dst:
        shutil.copyfileobj(resp, dst)
    if hashlib.sha256(tmp.read_bytes()).hexdigest() != cloudflared_sha256:
        tmp.unlink(missing_ok=True)
        raise RuntimeError("cloudflared checksum verification failed.")
    os.replace(tmp, cloudflared_path)
cloudflared_path.chmod(0o755)
clear_output()
print("NoRegretsRVC Streamlit setup complete! Model: NoRegretsVoice (32000 · HiFi-GAN)")


## 2. Launch Streamlit WebUI & Live Monitor


In [ ]:
import os
import re
import secrets
import subprocess
import sys
import time
from collections import deque
from pathlib import Path
from IPython.display import clear_output

Tunnel = "Gradio + Cloudflare" #@param ["Gradio + Cloudflare", "Horizon"]
horizon_id = "" #@param {type:"string"}

repo_dir = Path("/kaggle/working/program_ml")
cloudflared_path = repo_dir.parent / "cloudflared"
py_exec = "/kaggle/.venv/bin/python"

app_user = "noregrets"
app_pass = secrets.token_urlsafe(24)

cf_pattern = r"https://[a-zA-Z0-9-]+\.trycloudflare\.com"
hz_pattern = r"https://[a-zA-Z0-9.-]+\.hrzn\.run"
tunnel_procs = []
tunnel_logs = []
app_proc = None

def start_tunnel(cmd, name, pattern):
    out_path = repo_dir / f"{name}.log"
    f = open(out_path, "w", encoding="utf-8")
    proc = subprocess.Popen(cmd, stdout=f, stderr=subprocess.STDOUT, text=True)
    tunnel_procs.append(proc)
    tunnel_logs.append(f)
    deadline = time.time() + 60
    while time.time() < deadline:
        txt = out_path.read_text(encoding="utf-8", errors="replace") if out_path.exists() else ""
        m = re.search(pattern, txt)
        if m:
            return m.group(0)
        time.sleep(1)
    raise RuntimeError(f"Tunnel {name} failed to emit URL")

os.chdir(repo_dir)
subprocess.run(["fuser", "-k", "6969/tcp"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

if Tunnel == "Horizon":
    if not horizon_id.strip():
        raise ValueError("Enter your horizon_id before running Horizon tunnel.")
    subprocess.run(["npm", "install", "-g", "@hrzn/cli"], check=True)
    subprocess.run(["hrzn", "login", horizon_id], check=True)

# --- NoRegretsRVC Kaggle Live Monitor (spaced rows + mid-epoch + Speed|Step|Epoch table) ---
BOLD = "\033[1m"
RESET = "\033[0m"
ANSI_RE = re.compile(r"\x1B\[[0-?]*[ -/]*[@-~]")
EPOCH_RE = re.compile(r"epoch=(\d+)\s*\|\s*step=(\d+).*?training_speed=(\d+:\d{2}:\d{2})(?:.*?lowest_value=([0-9.]+))?")
TQDM_RE = re.compile(r"(\d+)%\|[^|]*\|\s*(\d+)/(\d+)\s*\[([^,\]]+),\s*([0-9.]+\s*(?:it/s|s/it))")
LOCAL_URL_RE = re.compile(r"(?:Running on local URL|Local URL|Network URL):\s*(https?://\S+)")
NOISE_RE = re.compile(r"(FutureWarning|UserWarning|DeprecationWarning|RuntimeWarning|pkg_resources|warnings\.warn|oneDNN|tensorflow/core|computation_placer|numa_node|Successfully opened dynamic library|weights_only|HTTP Request:|Checking out|libpng warning|ALSA|jack server|You can now view your Streamlit)", re.IGNORECASE)
MILESTONE_RE = re.compile(r"(Starting preprocess|Preprocess completed|preprocessed successfully|Starting pitch extraction|Pitch extraction completed|Starting embedding extraction|Embedding extraction completed|Saved index file|Starting training|Saving checkpoint|Saving model|Training finished|Training completed|Index file.*generated|Using .* vocoder|Using .* for training)", re.IGNORECASE)
ERROR_START_RE = re.compile(r"(Traceback \(most recent call last\)|RuntimeError:|ValueError:|FileNotFoundError:|OSError:|ModuleNotFoundError:|ImportError:|CUDA out of memory|OutOfMemoryError|Exception:|\bERROR:)")

session_links = []
milestones = deque(maxlen=6)
errors = deque(maxlen=25)
epoch_rows = deque(maxlen=15)
seen_epochs = set()
live_mid_epoch = ""
last_tq_render = 0.0
tb_lines_seen = 0

def upsert_link(label, value):
    for idx, (k, _) in enumerate(session_links):
        if k.strip() == label.strip():
            session_links[idx] = (label, value)
            return
    session_links.append((label, value))

def render_monitor(status="Running"):
    w_speed = max(9, max((len(r[0]) for r in epoch_rows), default=0))
    w_step = max(8, max((len(r[1]) for r in epoch_rows), default=0))
    w_epoch = max(6, max((len(r[2]) for r in epoch_rows), default=0))
    lines = [
        f"{BOLD}NoRegretsRVC Live Monitor (Beta 0.0.0){RESET}  [{status}]",
        "=" * 64,
        "",
    ]
    if app_pass:
        lines.append(f"WebUI Login      : {app_user} / {app_pass}")
        lines.append("")
    for label, val in session_links:
        if val:
            lines.append(f"{label:<17}: {val}")
            lines.append("")
    lines.append("-" * 64)
    lines.append("")
    if milestones:
        for m in milestones:
            lines.append(f"[Stage] {m}")
        lines.append("")
    if live_mid_epoch:
        lines.append(f"{BOLD}Live Epoch:{RESET} {live_mid_epoch}")
        lines.append("")
    if epoch_rows:
        lines.append(f"{'Speed':<{w_speed}} | {'Step':<{w_step}} | {BOLD}{'Epoch':<{w_epoch}}{RESET} | Lowest Loss")
        lines.append(f"{'-' * w_speed}-+-{'-' * w_step}-+-{'-' * w_epoch}-+------------")
        for spd, stp, ep, low in epoch_rows:
            lines.append(f"{spd:<{w_speed}} | {stp:<{w_step}} | {BOLD}{ep:<{w_epoch}}{RESET} | {low}")
    elif not errors:
        lines.append("Waiting for training epochs...")
    if errors:
        lines.append("")
        lines.append(f"{BOLD}Errors:{RESET}")
        for err in errors:
            lines.append(f"  {err}")
    clear_output(wait=True)
    print("\n".join(lines), flush=True)

try:
    env = os.environ.copy()
    if app_pass:
        env["APPLIO_AUTH_USERNAME"] = app_user
        env["APPLIO_AUTH_PASSWORD"] = app_pass
    env.update({
        "PYTHONWARNINGS": "ignore",
        "PYTHONUNBUFFERED": "1",
        "TF_CPP_MIN_LOG_LEVEL": "3",
        "TORCH_CPP_LOG_LEVEL": "ERROR",
        "TOKENIZERS_PARALLELISM": "false",
    })

    if Tunnel == "Gradio + Cloudflare":
        cf_url = start_tunnel([str(cloudflared_path), "tunnel", "--no-autoupdate", "--protocol", "http2", "--url", "http://127.0.0.1:6969"], "webui-cf", cf_pattern)
        upsert_link("NoRegretsRVC URL", cf_url)
        upsert_link("Cloudflare URL", cf_url)
    else:
        web_url = start_tunnel(["hrzn", "tunnel", "http://localhost:6969"], "webui-hrzn", hz_pattern)
        upsert_link("NoRegretsRVC URL", web_url)

    app_cmd = [
        py_exec, "-u", "-m", "streamlit", "run", "streamlit_app.py",
        "--server.address", "0.0.0.0",
        "--server.port", "6969",
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
    ]
    render_monitor("Starting NoRegretsRVC WebUI")
    app_proc = subprocess.Popen(app_cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace", bufsize=1)
    assert app_proc.stdout is not None
    for raw_line in app_proc.stdout:
        line = ANSI_RE.sub("", raw_line).replace("\r", "").strip()
        if not line:
            continue
        loc_m = LOCAL_URL_RE.search(line)
        if loc_m:
            upsert_link("Local UI URL", loc_m.group(1))
            render_monitor("Ready")
            continue
        ep_m = EPOCH_RE.search(line)
        if ep_m:
            ep_num, stp_num, spd_str = ep_m.group(1), ep_m.group(2), ep_m.group(3)
            low_str = ep_m.group(4) or "-"
            key = (ep_num, stp_num)
            if key not in seen_epochs:
                seen_epochs.add(key)
                epoch_rows.append((spd_str, stp_num, ep_num, low_str))
                live_mid_epoch = ""
                render_monitor("Training")
            continue
        tq_m = TQDM_RE.search(line)
        if tq_m:
            now_ts = time.time()
            if now_ts - last_tq_render >= 2.0 or tq_m.group(1) == "100":
                last_tq_render = now_ts
                live_mid_epoch = f"{tq_m.group(1)}% ({tq_m.group(2)}/{tq_m.group(3)}) · {tq_m.group(4)} · {tq_m.group(5)}"
                render_monitor("Training")
            continue
        if NOISE_RE.search(line) or "%|" in line or "it/s]" in line or "s/it]" in line:
            continue
        if tb_lines_seen > 0:
            errors.append(line)
            tb_lines_seen = 0 if re.match(r"^[A-Za-z_]*(Error|Exception):", line) else (tb_lines_seen - 1)
            render_monitor("Error")
            continue
        if ERROR_START_RE.search(line):
            errors.append(line)
            if "Traceback" in line:
                tb_lines_seen = 25
            render_monitor("Error")
            continue
        if MILESTONE_RE.search(line):
            if not milestones or milestones[-1] != line:
                milestones.append(line)
                render_monitor("Running")
    app_proc.wait()
finally:
    for p in reversed(tunnel_procs):
        if p.poll() is None: p.terminate()
    if app_proc is not None and app_proc.poll() is None:
        app_proc.terminate()
    for f in tunnel_logs:
        f.close()
    subprocess.run(["fuser", "-k", "6969/tcp"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
